In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from osgeo import gdal
from osgeo import osr
import rasterio
from tqdm import tqdm

In [2]:
data = xr.open_dataset(r"D:\atmospheric rivers\data\data2016.nc")

In [3]:
data

<xarray.Dataset> Size: 2GB
Dimensions:         (valid_time: 12, pressure_level: 16, latitude: 721,
                     longitude: 1440)
Coordinates:
    number          int64 8B ...
  * valid_time      (valid_time) datetime64[ns] 96B 2016-01-01 ... 2016-12-01
  * pressure_level  (pressure_level) float64 128B 1e+03 975.0 ... 550.0 500.0
  * latitude        (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    expver          (valid_time) <U4 192B ...
Data variables:
    u               (valid_time, pressure_level, latitude, longitude) float32 797MB ...
    v               (valid_time, pressure_level, latitude, longitude) float32 797MB ...
    q               (valid_time, pressure_level, latitude, longitude) float32 797MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

In [4]:
lat = data['latitude']
lon = data['longitude']
# z = data['z']    # 重力加速度
u = data['u']    # 纬向风
v = data['v']    # 经向风
q = data['q']    # 比湿

In [6]:
# 检出数据结构
# 12为 12 个月份，16 为十六个压力等级，721 和 1440 为经纬度
# 时间为 2020
for i in [u, v, q]:
    arr = np.asarray(i)
    print(arr.shape)

(12, 16, 721, 1440)
(12, 16, 721, 1440)
(12, 16, 721, 1440)


In [7]:
# 构建经纬网
lonmin, latmax, lonmax, latmin = [lon.min(), lat.max(), lon.max(), lat.min()]
l_lat = len(lat)
l_lon = len(lon)
lon_ce = (lonmax - lonmin) / (l_lon - 1)
lat_ce = (latmax - latmin) / (l_lat - 1)
# 将变量转化成数组
# z_arr = np.asarray(z)
u_arr = np.asarray(u)
v_arr = np.asarray(v)
q_arr = np.asarray(q)

In [16]:
names = [f'2016-{i}' for i in range(1, 13)]
pd = 2500    # 每层压力差
g = 9.81    # 重力加速度
# 计算水汽通量 IVT
for i, name in tqdm(enumerate(names), desc='IVT计算', total=12, leave=False):
    part1 = np.zeros((q_arr.shape[2], q_arr.shape[3]))
    part2 = np.zeros((q_arr.shape[2], q_arr.shape[3]))
    for ps in range(15):
        part1 += ( q_arr[i, ps, :, :]*u_arr[i, ps, :, :] + q_arr[i, ps+1, :, :]*u_arr[i, ps+1, :, :]) * 0.5 * pd
        part2 += ( q_arr[i, ps, :, :]*v_arr[i, ps, :, :] + q_arr[i, ps+1, :, :]*v_arr[i, ps+1, :, :]) * 0.5 * pd
    part1 = ((1/g)*part1)**2
    part2 = ((1/g)*part2)**2
    IVT = (part1+part2)**(1/2)
    # print(np.nanpercentile(IVT, 99.99))
    IVT[IVT<=200] = np.nan
    
    # 导出TVT数据
    out_path = f"D:\\atmospheric rivers\\data\\水汽通量IVT数据\\{name}.tif"
    driver = gdal.GetDriverByName('GTiff')
    out_tif = driver.Create(out_path, l_lon, l_lat, 1, gdal.GDT_Float32)
    out_tif.GetRasterBand(1).SetNoDataValue(np.nan)
    geotransform = (lonmin, lon_ce, 0, latmax, 0, -lat_ce)
    out_tif.SetGeoTransform(geotransform)
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    out_tif.SetProjection(srs.ExportToWkt())
    out_tif.GetRasterBand(1).WriteArray(IVT)
    out_tif.FlushCache()
    del out_tif